# Run agents on Eval1 subset

This notebook runs the selected biomedical agent on the fixed Eval1 subset created on `create_eval1_subset-ipynb`

Input:

`project_subset.csv`

The goal is to:

- load the fixed subset of 30 Eval1 queries
- run the selected agent/model on each query
- compare the agent output with the Eval1 reference answer
- store the results in a CSV file for analysis

Output:

`data/results_agents.csv`

In [ ]:
import json
import time
from pathlib import Path
import pandas as pd
from rich.console import Console
from bio_agents.frameworks import FRAMEWORK_REGISTRY
from bio_agents.tasks.biomni_eval1.task import BiomniEval1Task

console = Console()

#### Auxiliary Functions

In [ ]:
def safe_json(value):
    try:
        return json.dumps(value, ensure_ascii=False)
    except Exception:
        return str(value)

#### Load the Eval1 subset

In [ ]:
input_path = Path("project_subset.csv")
subset_df = pd.read_csv(input_path)

print("Loaded rows:", len(subset_df))
subset_df.head()

#### Configuration

In [ ]:
framework = "biomni"
model = "llama3"

output_path = Path("results_agents.csv")

commercial_mode = False
timeout_seconds = 600

print("Framework:", framework)
print("Model:", model)
print("Output path:", output_path)

#### Run the agent

In [ ]:
runner = FRAMEWORK_REGISTRY[framework]()
results = []

console.print(
    f"[bold]Running {len(subset_df)} queries with framework={framework}, model={model}[/bold]"
)

for i, row in subset_df.iterrows():
    task_name = row["task_name"]
    task_instance_id = row["task_instance_id"]
    input_query = row["input_query"]
    dataset_eval1_answer = row["dataset_eval1_answer"]

    console.print(
        f"[cyan][{i + 1}/{len(subset_df)}][/cyan] "
        f"{task_name} | instance_id={task_instance_id}"
    )

    task = BiomniEval1Task(
        task_name=task_name,
        task_instance_id=task_instance_id,
        prompt=input_query,
    )

    task_input = task.get_input()
    tools = task.get_tools()

    start = time.perf_counter()

    try:
        agent_result = runner.run(
            task_input.prompt,
            tools,
            model,
            commercial_mode=commercial_mode,
            timeout_seconds=timeout_seconds,
        )

        duration_s = round(time.perf_counter() - start, 3)
        eval_result = task.evaluate(agent_result)

        results.append(
            {
                "sample_index": row["sample_index"],
                "framework": framework,
                "model": model,
                "task_name": task_name,
                "task_instance_id": task_instance_id,
                "input_query": input_query,
                "dataset_eval1_answer": dataset_eval1_answer,
                "agent_reasoning_trace": safe_json(agent_result.tool_calls),
                "agent_output_answer": agent_result.output,
                "score": eval_result.score,
                "passed": eval_result.passed,
                "duration_s": duration_s,
                "tool_calls_count": len(agent_result.tool_calls),
                "agent_metadata": safe_json(agent_result.metadata),
                "metrics": safe_json(eval_result.metrics),
                "eval_error": (
                    eval_result.metrics.get("eval_error", "")
                    if isinstance(eval_result.metrics, dict)
                    else ""
                ),
                "run_error": "",
            }
        )

    except Exception as exc:
        duration_s = round(time.perf_counter() - start, 3)

        results.append(
            {
                "sample_index": row["sample_index"],
                "framework": framework,
                "model": model,
                "task_name": task_name,
                "task_instance_id": task_instance_id,
                "input_query": input_query,
                "dataset_eval1_answer": dataset_eval1_answer,
                "agent_reasoning_trace": "",
                "agent_output_answer": "",
                "score": 0.0,
                "passed": False,
                "duration_s": duration_s,
                "tool_calls_count": 0,
                "agent_metadata": "{}",
                "metrics": "{}",
                "eval_error": "",
                "run_error": str(exc),
            }
        )

#### Save results to CSV


In [ ]:
results_df = pd.DataFrame(results)

results_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved results to:", output_path)
print("Rows:", len(results_df))
print("Columns:", results_df.columns.tolist())

In [ ]:
results_df = pd.read_csv(output_path)
results_df.head()

#### Main comparison table and summary statistics

This table shows the most important columns for evaluating agent performance:

- input query
- Eval1 reference answer
- agent reasoning trace
- agent output answer
- score
- passed status

In [ ]:
results_df[
    [
        "sample_index",
        "task_name",
        "input_query",
        "dataset_eval1_answer",
        "agent_reasoning_trace",
        "agent_output_answer",
        "score",
        "passed",
    ]
].head(10)

In [ ]:
total = len(results_df)
passed = results_df["passed"].sum()
avg_score = results_df["score"].mean()
avg_duration = results_df["duration_s"].mean()

print("Total queries:", total)
print("Passed:", passed)
print("Failed:", total - passed)
print("Average score:", round(avg_score, 3))
print("Average duration (s):", round(avg_duration, 3))